# Adversarial attack against closed-box model
1. Do some distil API classifier into a local network (might be a small network on top of CLIP)
2. Run adversarial attack against it (gradient-based)

In [45]:
import base64
import io
import json
import os
from hashlib import md5
from typing import Dict, Optional

import open_clip
import requests
import torch
from PIL import Image


In [35]:
TARGET_SERVER_URL = "https://wstud-mmml26-classification.web.webis.de"

In [36]:
session = requests.Session()


# {img_hash: response}

### Parsing ###

def parse_img(img: Image.Image) -> tuple[str, str]:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    img_base64_bytes = base64.b64encode(buf.getvalue())
    img_base64 = img_base64_bytes.decode("utf-8")
    img_hash = md5(img_base64_bytes).hexdigest()
    return img_base64, img_hash


### Cache ###

def get_cache(img_hash) -> Optional[Dict]:
    with open("cache.json", "r") as f:
        cache_content = json.load(f)
        return cache_content.get(img_hash, None)


def set_cache(img_hash, api_response) -> None:
    with open("cache.json", "r") as f:
        cache_content = json.load(f)
        cache_content[img_hash] = api_response
    with open("cache.json", "w") as f:
        json.dump(cache_content, f)


### Network ###

def query_api(img_base64) -> Dict:
    try:
        resp = session.post(
            TARGET_SERVER_URL + "/predict",
            json={"image": img_base64, "classes": ["class1", "class2", "class3", "class4"]},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"Request failed {e}")
        return {}


### User call ###

def get_score(img) -> Dict:
    img_base64, img_hash = parse_img(img)
    cache_hit = get_cache(img_hash)
    if cache_hit:
        return cache_hit
    api_response = query_api(img_base64)
    set_cache(img_hash, api_response)
    return api_response

In [37]:
dir_path = "img"
images = [Image.open(f"{dir_path}/{p}") for p in os.listdir(dir_path)]

In [55]:
for i in images:
    score = get_score(i)
    print(score)

{'classes': {'class1': 0.3910961449146271, 'class2': 0.6089038848876953, 'class3': 0.9323794841766357, 'class4': 0.06762055307626724}}
{'classes': {'class1': 0.11249769479036331, 'class2': 0.8875023126602173, 'class3': 0.9596433043479919, 'class4': 0.04035666584968567}}
{'classes': {'class1': 0.1592283844947815, 'class2': 0.8407716155052185, 'class3': 0.3030906319618225, 'class4': 0.6969093680381775}}
{'classes': {'class1': 0.8734389543533325, 'class2': 0.12656109035015106, 'class3': 0.5671042799949646, 'class4': 0.4328957200050354}}
{'classes': {'class1': 0.9718536138534546, 'class2': 0.02814633585512638, 'class3': 0.38879862427711487, 'class4': 0.6112013459205627}}


In [54]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer('ViT-B-32')

classes = ["a dog", "a cat", "happy", "sad"]
for image in images:
    image = preprocess(image).unsqueeze(0)
    text = tokenizer(classes)

    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

    for i in range(len(classes)):
        print(f"{classes[i]}: {text_probs[0][i]}\t", end="")
    print("")

a dog: 0.00018388441822025925	a cat: 0.0018814863869920373	happy: 0.44413119554519653	sad: 0.5538034439086914	
a dog: 0.00020033727923873812	a cat: 0.00011502997222123668	happy: 0.20073115825653076	sad: 0.7989534735679626	
a dog: 0.0033288365229964256	a cat: 0.0002620428567752242	happy: 0.6809172630310059	sad: 0.31549182534217834	
a dog: 0.009141744114458561	a cat: 0.001134541817009449	happy: 0.02457827515900135	sad: 0.965145468711853	
a dog: 0.00035396000021137297	a cat: 6.542443315993296e-06	happy: 0.2608000636100769	sad: 0.7388394474983215	
